# 2. Conventional Approaches to Phase Identification

In [ ]:
# @title Environment Setup
!pip install pymatgen numpy matplotlib scipy -q
print("Packages installed")

In [ ]:
# @title Load Tutorial Data
import os

REPO = "MRS_CH08_Tutorial"
REPO_URL = "https://github.com/Szymanski-Group/MRS_CH08_Tutorial.git"

if os.path.basename(os.getcwd()) == REPO:
    print("Data already present")
elif os.path.exists(REPO):
    os.chdir(REPO)
    print("Data already present")
else:
    !git clone {REPO_URL} -q
    os.chdir(REPO)
    print("Data loaded successfully")


## 2a) Search Match

Here we detect peaks in experimental patterns and rank candidate phases using FoM-style line matching scores.

In [ ]:
import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from tutorial_utils.runners import run_search_match
from tutorial_utils.sections import conventional_search_match as s02a


def create_search_match_demo(
    top_k_to_print=3,
    max_experiment_patterns=4,
    output_dir="outputs/conventional/search_match",
    show_steps=True,
):
    """Run the Search-Match demo with configurable settings."""
    return run_search_match(
        top_k_to_print=top_k_to_print,
        max_experiment_patterns=max_experiment_patterns,
        output_dir=output_dir,
        show_steps=show_steps,
    )


def inspect_search_match_steps(
    pattern_file="data/exp_patterns/one_phase/In2O3.xy",
    reference_dir="data/reference_structures",
    top_k=5,
    show_plot=True,
):
    """Expose the key inner steps: preprocess -> peak detection -> ranking."""
    tt, intensity = s02a.load_pattern(Path(pattern_file))
    _, obs_peaks = s02a.detect_peaks(tt, intensity)
    refs = s02a.load_reference_library(sorted(Path(reference_dir).glob("*.cif")))
    by_dewolff, by_smith = s02a.rank_phases(obs_peaks, refs)

    print(f"Step 1: loaded profile with {len(tt)} points")
    print(f"Step 2: detected {len(obs_peaks)} peaks")
    print(f"Step 3: ranked {len(refs)} candidate phases")

    print("\nTop by de Wolff:")
    for i, row in enumerate(by_dewolff[:top_k], start=1):
        print(f"  {i}. {row['phase']:<16s} score={row['de_wolff']:.3f}")

    if show_plot:
        plt.figure(figsize=(8, 3.8))
        plt.plot(tt, intensity, color="black", linewidth=1.8, label="Experimental")
        plt.scatter(obs_peaks, np.interp(obs_peaks, tt, intensity), s=22, color="#dc2626", label="Detected peaks")
        plt.xlim(s02a.PLOT_MIN_ANGLE, s02a.PLOT_MAX_ANGLE)
        plt.ylim(0, 105)
        plt.xlabel("2θ")
        plt.ylabel("Intensity")
        plt.title("Search-Match inner step: detected peaks")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return {
        "two_theta": tt,
        "intensity": intensity,
        "detected_peaks": obs_peaks,
        "top_by_dewolff": by_dewolff[:top_k],
        "top_by_smith_snyder": by_smith[:top_k],
    }


## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
# @title Run Search-Match Demo
create_search_match_demo()

# Try on your own:
# create_search_match_demo(top_k_to_print=5, max_experiment_patterns=None)
# inspect_search_match_steps(pattern_file="data/exp_patterns/one_phase/TiO2.xy")


## What To Observe
Compare top-ranked phases and where the reference sticks align with experimental peaks.

In [ ]:
for p in sorted(glob.glob("outputs/conventional/search_match/*_summary.png"))[:3]:
    display(Image(p))

## Summary
- Peak-list matching is fast and interpretable.
- Performance depends on peak detection quality and tolerance settings.
- This method can struggle when peaks overlap or broaden heavily.

## Next Steps
Continue to the next section below in this notebook.

## 02b - Profile Correlation

In [ ]:
import glob
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Image, display

from tutorial_utils.runners import run_profile_correlation
from tutorial_utils.sections import conventional_profile_correlation as s02b


def create_profile_correlation_demo(
    top_k_to_print=3,
    max_experiment_patterns=4,
    fwhm=0.30,
    gauss_frac=0.2,
    output_dir="outputs/conventional/profile_correlation",
    show_steps=True,
):
    """Run full-profile correlation with configurable broadening settings."""
    return run_profile_correlation(
        top_k_to_print=top_k_to_print,
        max_experiment_patterns=max_experiment_patterns,
        fwhm=fwhm,
        gauss_frac=gauss_frac,
        output_dir=output_dir,
        show_steps=show_steps,
    )


def inspect_profile_correlation_steps(
    pattern_file="data/exp_patterns/one_phase/In2O3.xy",
    reference_dir="data/reference_structures",
    top_k=5,
    show_plot=True,
):
    """Expose key inner steps: baseline removal -> profile simulation -> similarity ranking."""
    two_theta, exp_profile = s02b.load_experimental_profile(Path(pattern_file))
    ref_lib = s02b.load_reference_stick_library(sorted(Path(reference_dir).glob("*.cif")))
    by_pearson, by_cosine, simulated = s02b.rank_phases(exp_profile, two_theta, ref_lib)

    print(f"Step 1: loaded profile with {len(two_theta)} points")
    print(f"Step 2: simulated {len(ref_lib)} reference profiles")
    print("Step 3: ranked by Pearson and cosine similarity")

    print("\nTop by Pearson:")
    for i, row in enumerate(by_pearson[:top_k], start=1):
        print(f"  {i}. {row['phase']:<16s} score={row['pearson']:.3f}")

    best_phase = by_pearson[0]["phase"]
    if show_plot:
        plt.figure(figsize=(8, 3.8))
        plt.plot(two_theta, exp_profile, color="black", linewidth=1.8, label="Experimental")
        plt.plot(two_theta, simulated[best_phase], color="#1f4ed8", linewidth=1.8, label=f"Best match: {best_phase}")
        plt.xlim(s02b.MIN_ANGLE, s02b.MAX_ANGLE)
        plt.ylim(0, 105)
        plt.xlabel("2θ")
        plt.ylabel("Intensity")
        plt.title("Profile-correlation inner step: best simulated match")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return {
        "two_theta": two_theta,
        "experimental_profile": exp_profile,
        "top_by_pearson": by_pearson[:top_k],
        "top_by_cosine": by_cosine[:top_k],
        "simulated_profiles": simulated,
    }


# 02b — Full-Profile Correlation

Instead of line matching, this method compares full simulated and observed profiles via correlation metrics.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
# @title Run Profile-Correlation Demo
create_profile_correlation_demo()

# Try on your own:
# create_profile_correlation_demo(fwhm=0.45, gauss_frac=0.4)
# inspect_profile_correlation_steps(pattern_file="data/exp_patterns/one_phase/TiO2.xy")


## What To Observe
Look for differences between Pearson and cosine rankings on the same pattern.

In [ ]:
for p in sorted(glob.glob("outputs/conventional/profile_correlation/*_profile-correlation.png"))[:3]:
    display(Image(p))

## Summary
- Profile-level comparison uses more information than discrete peak lists.
- Pearson and cosine can prioritize slightly different candidates.
- Baseline handling strongly influences correlation scores.

## Next Steps
Continue to the next section below in this notebook.

## 02c - Rietveld Refinement

In [ ]:
import glob
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Image, display

from tutorial_utils.runners import run_rietveld_sequential
from tutorial_utils.sections import conventional_rietveld as s02c


def create_rietveld_demo(
    top_k_to_print=3,
    patterns_to_run=("TiO2", "ZrO2"),
    background_degree=6,
    fwhm_init=0.30,
    output_dir="outputs/conventional/rietveld",
    show_steps=True,
):
    """Run sequential Rietveld-style refinement with configurable arguments."""
    return run_rietveld_sequential(
        top_k_to_print=top_k_to_print,
        patterns_to_run=patterns_to_run,
        background_degree=background_degree,
        fwhm_init=fwhm_init,
        output_dir=output_dir,
        show_steps=show_steps,
    )


def inspect_rietveld_steps(
    pattern_file="data/exp_patterns/one_phase/TiO2.xy",
    phase_name="TiO2_136",
    reference_dir="data/reference_structures",
    show_plot=True,
):
    """Expose key inner steps: background -> lattice -> width refinement for one phase."""
    two_theta, y_obs = s02c.load_experimental_profile(Path(pattern_file))
    structures = s02c.load_reference_structures(sorted(Path(reference_dir).glob("*.cif")))

    if phase_name not in structures:
        same_formula = [k for k in structures if k.startswith(phase_name.split("_", 1)[0])]
        phase_name = same_formula[0] if same_formula else sorted(structures)[0]

    calculator = s02c.XRDCalculator(wavelength=s02c.WAVELENGTH)
    result = s02c.refine_phase_sequential(two_theta, y_obs, structures[phase_name], calculator)

    rwp_step1 = s02c.compute_rwp(y_obs, result["y_fit_step1"])
    rwp_step2 = s02c.compute_rwp(y_obs, result["y_fit_step2"])
    rwp_final = result["rwp"]

    print(f"Phase inspected: {phase_name}")
    print(f"Step 1 (background) Rwp: {rwp_step1:.2f}%")
    print(f"Step 2 (lattice)    Rwp: {rwp_step2:.2f}%")
    print(f"Step 3 (width)      Rwp: {rwp_final:.2f}%")
    print(f"Final Pearson: {result['pearson']:.3f}")
    print(f"Final scales (a,b,c): {result['scales']}")
    print(f"Final FWHM: {result['fwhm']:.4f}")

    if show_plot:
        plt.figure(figsize=(8, 4.0))
        plt.plot(two_theta, y_obs, color="black", linewidth=2.0, label="Experimental")
        plt.plot(two_theta, result["y_fit_step1"], color="#1f4ed8", linewidth=1.2, label="Step 1 fit")
        plt.plot(two_theta, result["y_fit_step2"], color="#7e22ce", linewidth=1.2, label="Step 2 fit")
        plt.plot(two_theta, result["y_fit_final"], color="#dc2626", linewidth=1.8, label="Final fit")
        plt.xlim(s02c.MIN_ANGLE, s02c.MAX_ANGLE)
        plt.xlabel("2θ")
        plt.ylabel("Intensity")
        plt.title("Rietveld inner steps for one candidate phase")
        plt.legend()
        plt.tight_layout()
        plt.show()

    out = dict(result)
    out["two_theta"] = two_theta
    out["y_obs"] = y_obs
    out["phase"] = phase_name
    return out


# 02c — Sequential Rietveld-Style Refinement

This simplified workflow refines background, lattice scales, and width parameters in sequence for each candidate phase.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
# @title Run Sequential Rietveld Demo
create_rietveld_demo()

# Try on your own:
# create_rietveld_demo(patterns_to_run=("LiMnO2",), background_degree=4)
# inspect_rietveld_steps(pattern_file="data/exp_patterns/one_phase/ZrO2.xy", phase_name="ZrO2_14")


## What To Observe
Track how each refinement stage improves the fit and lowers Rwp.

In [ ]:
for p in sorted(glob.glob("outputs/conventional/rietveld/*_rietveld-sequential.png")):
    display(Image(p))

## Summary
- Sequential refinement isolates effects of key parameter groups.
- Rwp provides a compact fit-quality ranking.
- Refinement-based methods are accurate but more compute-intensive.

## Next Steps
Continue to **03 — ML + Deep Learning**: [Open in Colab](https://colab.research.google.com/github/Szymanski-Group/MRS_CH08_Tutorial/blob/main/notebooks/03_ML-Deep-Learning.ipynb)